In [1]:
import sys
sys.path.insert(0, "/Users/wyleong/Desktop/Python Projects/Singapore-Gov-Procurement-GeBiz")

In [18]:
pip install backend

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sqlite3
from backend.services.analytics import *
from backend.config import DB_PATH

# Connect to the database
conn = sqlite3.connect(DB_PATH)

# 1. KPI cards — medical tenders only
market_overview(conn)
# {'total_awarded': 6185210570, 'num_tenders': 1124, 'num_agencies': 54, 'num_suppliers': 851}

# 2. Spend by quarter
spend_over_time(conn, period="quarterly")

# Option 2: capture and print nicely
data = spend_over_time(conn, period="quarterly")
for row in data:
    print(f"{row['period']}: ${row['total_awarded']:,.0f}  ({row['num_tenders']} tenders)")


# 3. Top 5 agencies
top_agencies(conn, limit=5)

# 4. Top 10 suppliers in diagnostics only
top_suppliers(conn, limit=10, category="diagnostics")

# 5. Search for "gloves" tenders above $100K
search_tenders(conn, keyword="gloves", min_amount=100_000)

# 6. All functions accept filters — combine as needed
top_suppliers(conn, limit=5, agency="Health Sciences Authority", date_from="2023-01-01")

# 7. All tenders (not just medical)
market_overview(conn, medical_only=False)

conn.close()


NameError: name 'r_time' is not defined

In [12]:
import pandas as pd
import requests

DATASET_ID = "d_acde1106003906a75c3fa052592f2fcb"
BASE_URL = "https://data.gov.sg/api/action/datastore_search"
PAGE_SIZE = 1000  # max per call

def fetch_all_records(dataset_id: str = DATASET_ID,
                      page_size: int = PAGE_SIZE) -> pd.DataFrame:
    all_records = []
    offset = 0

    while True:
        resp = requests.get(
            BASE_URL,
            params={
                "resource_id": dataset_id,
                "limit": page_size,
                "offset": offset,
            },
        )
        resp.raise_for_status()
        payload = resp.json()

        # defensive: check success and error message
        if not payload.get("success", False):
            raise RuntimeError(f"API error: {payload.get('error')}")

        result = payload["result"]
        records = result.get("records", [])

        if not records:
            # no more data
            print(f"\nNo records at offset {offset}, stopping.")
            break

        all_records.extend(records)
        print(f"\rFetched {len(all_records)} records so far...", end="")

        # if we got fewer than a full page, we’ve hit the last page
        if len(records) < page_size:
            break

        offset += page_size

    print("\nDone.")
    return pd.DataFrame(all_records)

# Use it
df = fetch_all_records()
print(df.shape)
df.head()

Fetched 18021 records so far...
Done.
(18021, 8)


,_id,tender_no,tender_description,agency,award_date,tender_detail_status,supplier_name,awarded_amt
0,1,ACR000ETT20300002,INVITATION TO TENDER FOR THE PROVISION OF SERV...,Accounting And Corporate Regulatory Authority,10/11/2020,Awarded by Items,DELOITTE & TOUCHE ENTERPRISE RISK SERVICES PTE...,285000
1,2,ACR000ETT20300002,INVITATION TO TENDER FOR THE PROVISION OF SERV...,Accounting And Corporate Regulatory Authority,10/11/2020,Awarded by Items,KPMG SERVICES PTE. LTD.,90000
2,3,ACR000ETT20300003,PROVISION OF AN IT SECURITY CONTROLS AND OPERA...,Accounting And Corporate Regulatory Authority,9/12/2020,Awarded to Suppliers,ERNST & YOUNG ADVISORY PTE. LTD.,182400
3,4,ACR000ETT20300004,"CONCEPTUALIZATION, DESIGN, BUILD, SET-UP OF NE...",Accounting And Corporate Regulatory Authority,9/3/2021,Awarded to Suppliers,D' PERCEPTION SINGAPORE PTE. LTD.,3071056.4
4,5,ACR000ETT21000001,"DESIGN, DEVELOPMENT, CUSTOMIZATION, DELIVERY, ...",Accounting And Corporate Regulatory Authority,6/9/2021,Awarded to Suppliers,ALPHA ZETTA PTE. LTD.,2321600


In [ ]:
from pathlib import Path

# notebook is in /notebooks, project root is one level up
PROJECT_ROOT = Path.cwd().parent

output_path = PROJECT_ROOT /"data"/"GovernmentProcurementviaGeBIZ.csv"
df.to_csv(output_path, index=False)

print(f"Saved {len(df)} rows to {output_path}")

# Narrow down to life science related agencies

In [23]:
core_health = [
    "Ministry of Health-Ministry Headquarter",
    "Health Promotion Board",
    "Health Sciences Authority",
    "Singapore Medical Council",
    "Singapore Nursing Board",
]

env_food_water = [
    "Ministry of Sustainability and the Environment",
    "National Environment Agency",
    "Singapore Food Agency",
    "National Parks Board",
    "Public Utilities Board",
]

education_research = [
    "Agency for Science, Technology and Research",
    "Science Centre Board",
    "Institute of Technical Education",
    "Ngee Ann Polytechnic",
    "Nanyang Polytechnic",
    "Republic Polytechnic",
    "Singapore Polytechnic",
    "Temasek Polytechnic",
    "School of Science and Technology, Singapore",
    "Ministry of Education",
    "Ministry of Education - Schools",
]

workplace_health_safety = [
    "Ministry of Manpower - Occupational Safety & Health Division",
    "Ministry of Manpower-Ministry Headquarter",
]

social_care_adjacent = [
    "National Council of Social Service",
    "Yellow Ribbon Singapore",
    "Majlis Ugama Islam Singapura",
]

In [24]:
from itertools import chain

agency_segment_map = {}

agency_segment_map.update({a: "core_health" for a in core_health})
agency_segment_map.update({a: "env_food_water" for a in env_food_water})
agency_segment_map.update({a: "education_research" for a in education_research})
agency_segment_map.update({a: "workplace_health_safety" for a in workplace_health_safety})
agency_segment_map.update({a: "social_care_adjacent" for a in social_care_adjacent})

df["agency_segment"] = df["agency"].map(agency_segment_map).fillna("other")

lifesci_segments = [
    "core_health",
    "env_food_water",
    "education_research",
    "workplace_health_safety",
    "social_care_adjacent",
]

lifesci_df = df[df["agency_segment"].isin(lifesci_segments)].copy()

print("Total tenders (life-sci inclusive view):", len(lifesci_df))
print("Distinct agencies:", lifesci_df["agency"].nunique())
print(lifesci_df["agency_segment"].value_counts())

lifesci_suppliers = (
    lifesci_df["supplier_name"]
    .dropna()
    .unique()
    .tolist()
)
len(lifesci_suppliers)

Total tenders (life-sci inclusive view): 6710
Distinct agencies: 26
agency_segment
education_research         3325
env_food_water             1996
core_health                1154
social_care_adjacent        148
workplace_health_safety      87
Name: count, dtype: int64


2737

# We try to use statistics to find keywords that match the tender description we are looking for...

In [30]:
s = (
    lifesci_df["tender_description"]
    .dropna()
    .astype(str)
    .str.strip()
)

num_unique = s.nunique()
print("Unique descriptions:", num_unique)

Unique descriptions: 4487


In [39]:
competitor_keywords = [
    'scientific', 'laboratories', 'laboratory', 'bioscience', 'biosciences',
    'chemical', 'instrument', 'analytical', 'biotech', 'biological',
    'chromatography', 'genomics', 'diagnostic', 'pharma', 'reagent',
    'biosystems', 'bios', 'medical', 'biomed', 'gene'
]

known_competitors = [
    'Merck', 'Sigma Aldrich', 'Avantor', 'VWR', 'LIFE TECHNOLOGIES',
    'Agilent', 'PerkinElmer', 'Shimadzu', 'Waters', 'GE Healthcare',
    'Bio-Rad', 'Qiagen', 'Illumina', 'Becton Dickinson', 'BD Diagnostics',
    'Beckman Coulter', 'Thermo Fisher', 'Thermo Scientific', 'Axil Scientific',
    'SPD Scientific', 'ALtec Instruments', 'Eppendorf', 'Sartorius', 'Horiba',
    'Zeiss', 'Olympus', 'EXBIO', 'FUJIFILM', 'Hettich', 'Hitachi', 'Roche',
    'Abbott', 'Analytik Jena', 'Mettler Toledo', 'Metrohm', 'Kuhner', 'Lonza',
    'Promega', 'Takara', 'BioNex', 'Labquip', 'Labchem', 'Lab Science',
    'Stryker', 'Medtronic', 'STORZ',  # <- need comma here or Python concatenates strings
    'Philips Healthcare', 'Siemens health', 'Canon Medical Systems',
    'Fujifilm Medical Systems', 'Olympus Medical Systems', 'Sakura Finetek',
    'Sysmex',
]
# 'Roche', 'Abbott', 'Beckman Coulter' are duplicated but that's harmless


import re
import numpy as np

sup = df["supplier_name"].fillna("").astype(str)

# lowercased for case-insensitive search
sup_lower = sup.str.lower()

# pattern for generic keywords
kw_pattern = "|".join(re.escape(k.lower()) for k in competitor_keywords)

mask_kw = sup_lower.str.contains(kw_pattern, na=False)

# pattern for known competitor brands (also case-insensitive)
brand_pattern = "|".join(re.escape(b.lower()) for b in known_competitors)

mask_brand = sup_lower.str.contains(brand_pattern, na=False)

df["is_competitor_supplier"] = mask_kw | mask_brand

df_comp = df[df["is_competitor_supplier"]].copy()
print("Tenders with competitor-like suppliers:", len(df_comp))
print("Unique such suppliers:", df_comp["supplier_name"].nunique())


Tenders with competitor-like suppliers: 860
Unique such suppliers: 185


In [40]:
(df_comp["supplier_name"]
 .value_counts()
 .head(50))

supplier_name
AGILENT TECHNOLOGIES SINGAPORE (SALES) PTE. LTD.    46
LIFE TECHNOLOGIES HOLDINGS PTE. LTD.                24
THERMO FISHER SCIENTIFIC PTE. LTD.                  23
BIOMED DIAGNOSTICS PTE LTD                          22
ZUELLIG PHARMA PTE. LTD.                            21
VWR SINGAPORE PTE. LTD.                             21
ITS SCIENCE & MEDICAL PTE LTD                       21
RAFFLES MEDICAL GROUP LTD                           20
SPD SCIENTIFIC PTE LTD                              19
FISHER SCIENTIFIC PTE LTD                           19
CHEMICAL INDUSTRIES (FAR EAST) LIMITED.             18
BIO-RAD LABORATORIES (SINGAPORE) PTE LTD            17
BIO LABORATORIES PTE LTD                            17
MERCK PTE. LTD.                                     17
YEAP MEDICAL SUPPLIES PTE. LTD.                     16
WATERS PACIFIC PTE. LTD.                            16
SHIMADZU (ASIA PACIFIC) PTE LTD                     15
RESEARCH INSTRUMENTS PTE LTD                       

In [49]:
import re

# === 3.1: Build a life-sci keyword list from your n-gram inspection ===

# Single-word tokens that are clearly life-sci / diagnostic / lab-ish
lifesci_unigrams = [
    "laboratory", "laboratories", "lab", "diagnostic",
    "spectrometer", "spectrometry", "chromatography", "chromatograph",
    "uhplc", "hplc", "icp", "hrms", "mass",  # will be combined in phrases anyway
    "centrifuge", "centrifuges", "microscope", "cytometry", "cytometer",
    "reagent", "reagents", "antibody", "antibodies", "consumables",
    "pipette", "pipettes", "pipettor", "micropipette", "tips", "tubes",
    "serology", "biorisk", "nucleic", "dna", "rna", "sequencing",
    "vaccination", "vaccine", "screening", "swab",
    "medical", "healthcare", "clinic", "clinical",
]

# Multi-word phrases taken directly from your n-gram output (or obvious variants)
lifesci_phrases = [
    # lab equipment / instrumentation
    "laboratory equipment",
    "laboratory analytical",
    "analytical instruments",
    "analytical instruments for",
    "mass spectrometer",
    "mass spectrometer system",
    "quadrupole mass spectrometer",
    "high resolution mass",
    "hrms coupled",
    "hrms coupled with",
    "ultra high performance",
    "high performance liquid",
    "performance liquid chromatography",
    "liquid chromatography system",
    "chromatography system",
    "gas chromatograph",
    "gas chromatograph liners",
    "spectrometer system",
    "slide scanner",

    # lab supplies & consumables
    "laboratory consumables",
    "laboratory chemicals",
    "laboratory reagents",
    "laboratory supplies",
    "laboratory analytical instruments",
    "chemicals and supplies",
    "chemicals consumables",
    "reagents and consumables",
    "reagents and kits",
    "centrifuge tubes",
    "pipette tips",
    "pipette tips for",
    "centrifuge tubes for",
    "dehydrated culture media",
    "culture media",
    "culture isolation",
    "culture isolation rapid",
    "nursing laboratory consumables",
    "nursing consumables",
    "nursing laboratories",

    # diagnostics / kits
    "covid vaccination",
    "covid vaccination services",
    "covid testing",
    "covid testing covid",
    "covid antigen",
    "covid antigen rapid",
    "rapid detection kits",
    "detection kits",
    "test kits",
    "medical examination services",
    "medical services",
    "diagnostic kit",
    "serology testing",
    "typing kits",

    # flow / cell / imaging
    "flow cytometry",
    "cytometry cell",
    "cytometry cell analysers",
    "units flow cytometry",
    "flow cytometer",
    "imaging system",
    "cell culture",

    # biosafety / biorisk / metrology
    "laboratory biorisk management",
    "biorisk management",
    "chemical metrology laboratory",
    "chemical metrology",

    # programmes clearly health-related
    "school health sciences",
    "health sciences ngee",
    "health sciences authority",
    "health promotion board",
    "covid vaccination and",
    "screening and vaccination",
    "health screening",
    "health screening and",
    "youth preventive",
    "youth preventive maintenance",  # from output; keep as is
]

# === 3.2: Build regex patterns ===

# Unigrams as word-boundary matches
unigram_pattern = r"\b(" + "|".join(re.escape(w.lower()) for w in lifesci_unigrams) + r")\b"

# Phrases as simple substring matches (already multi-word)
phrase_pattern = "|".join(re.escape(p.lower()) for p in lifesci_phrases)

# Combined pattern
full_pattern = "(" + unigram_pattern + "|" + phrase_pattern + ")"

# === 3.3: Apply to ALL tenders ===

desc = df["tender_description"].fillna("").str.lower()

df["has_lifesci_desc_kw"] = desc.str.contains(full_pattern, na=False)

print("Tenders with life-sci description keywords:",
      df["has_lifesci_desc_kw"].sum())

/var/folders/8p/q2rwlbfj7j59ht09w0rq8bh80000gn/T/ipykernel_78965/1129567709.py:122: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["has_lifesci_desc_kw"] = desc.str.contains(full_pattern, na=False)


Tenders with life-sci description keywords: 1639


In [50]:
# Optional: if you also have agency_segment + lifesci_segments, include them:
lifesci_segments = [
    "core_health",
    "env_food_water",
    "education_research",
    "workplace_health_safety",
    "social_care_adjacent",
]

has_lifesci_agency = df.get("agency_segment", "").isin(lifesci_segments) \
    if "agency_segment" in df.columns else False

# Final rule: life-sci tender if ANY of these are true
df["is_lifesci_tender"] = (
    df["is_competitor_supplier"] |
    df["has_lifesci_desc_kw"] |
    has_lifesci_agency
)

lifesci_tenders = df[df["is_lifesci_tender"]].copy()

print("Final life-sci tenders:", len(lifesci_tenders))

Final life-sci tenders: 6957


Tender Classification END

In [13]:

summary = df_comp.groupby(['supplier_name','agency']).agg(
    num_awards=('tender_no','count'),
    total_value=('awarded_amt','sum')
).reset_index()
summary.head()

,supplier_name,agency,num_awards,total_value
0,1 BISHAN MEDICAL PTE. LTD.,Health Sciences Authority,1,910000.0
1,1 BISHAN MEDICAL PTE. LTD.,Ministry of Education,1,1.0
2,57 MEDICAL PTE. LTD.,Ministry of Health-Ministry Headquarter,1,2360000.0
3,ABBOTT LABORATORIES (SINGAPORE ) PRIVATE LIMITED,Health Sciences Authority,1,9095020.0
4,ABSOLUTE INSTRUMENT SYSTEMS (PTE.) LTD.,National Environment Agency,3,378848.0


In [14]:
import plotly.express as px
import plotly.io as pio
import copy
# Consolidate total_value by summing across all agencies for each supplier
top_50_suppliers = (
    summary.groupby('supplier_name', as_index=False)
    .agg({'total_value': 'sum'})
    .sort_values(by='total_value', ascending=False)
    .head(50)
)


# Create a stacked bar chart grouped by agency
# -----------------------------------------------------
# 2. “Apple” template
# Swap clone() for copy.deepcopy()
apple = copy.deepcopy(pio.templates["simple_white"])

apple.layout.font = dict(
    family="Helvetica Neue, Arial, sans-serif",
    color="#1d1d1f",
    size=14,
)
apple.layout.title = dict(font=dict(size=24, family="Helvetica Neue"))
apple.layout.colorway = ["#007aff"]

# subtle grid
for ax in ("xaxis", "yaxis"):
    getattr(apple.layout, ax).update(
        gridcolor="#e5e5e7",
        zeroline=False,
        showline=False,
    )

pio.templates["apple"] = apple  # register

# -----------------------------------------------------
# 3. Build chart
fig = px.bar(
    top_50_suppliers,
    y="supplier_name",
    x="total_value",
    text="total_value",
    template="apple",
)

# -----------------------------------------------------
# 4. Final polish
fig.update_traces(
    texttemplate="%{text:,.0f}",     # thousands separator
    textposition="outside",
    marker_line_width=0,
)

fig.update_yaxes(
    title=None,
    categoryorder="total ascending",
)

fig.update_layout(
    title=dict(text="Top 50 Suppliers by Spend", x=0.02, xanchor="left"),
    xaxis_title="Total Value",
    bargap=0.18,
    showlegend=False,
    margin=dict(l=140, r=40, t=60, b=40),
    height=1000,
    width=900,
)

fig.show()

# Let's see the breakdown by Agency for each Supplier

In [15]:


# --- 1. Prerequisites -------------------------------------------------
# Make sure these three columns exist and are the right types
summary['supplier_name'] = summary['supplier_name'].astype(str)
summary['agency']        = summary['agency'].fillna('Unknown').astype(str)
summary['total_value']   = pd.to_numeric(summary['total_value'], errors='coerce').fillna(0)

# --- 2. Identify the 50 biggest suppliers (overall) -------------------
supplier_totals = (
    summary.groupby('supplier_name', as_index=False)['total_value']
           .sum()
           .sort_values('total_value', ascending=False)
           .head(50)
)

top_50_names = supplier_totals['supplier_name']

# --- 3. Keep only rows that belong to those suppliers -----------------
df_top50_detail = summary[summary['supplier_name'].isin(top_50_names)].copy()

# Preserve the ranking order for prettier plotting
df_top50_detail['supplier_name'] = pd.Categorical(
    df_top50_detail['supplier_name'],
    categories=supplier_totals['supplier_name'],  # already sorted desc
    ordered=True
)

df_top50_detail['value_m'] = (df_top50_detail['total_value'] / 1e6).round(2).astype(str) + ' M'


# --- 4. Plot: each bar = supplier, segments = agency ------------------
if "apple" not in pio.templates:
    apple = copy.deepcopy(pio.templates["simple_white"])
    apple.layout.font = dict(
        family="Helvetica Neue, Arial, sans-serif",
        color="#1d1d1f",
        size=14
    )
    apple.layout.title = dict(font=dict(size=24, family="Helvetica Neue"))
    # A restrained Apple-style categorical palette
    apple.layout.colorway = [
        "#007aff",  # blue
        "#34c759",  # green
        "#ff9f0a",  # orange
        "#af52de",  # purple
        "#ff375f",  # pink/red
        "#5e5ce6",  # indigo
    ]
    for ax in ("xaxis", "yaxis"):
        getattr(apple.layout, ax).update(
            gridcolor="#e5e5e7",
            zeroline=False,
            showline=False
        )
    pio.templates["apple"] = apple

# ------------------------------------------------------------------
# 1) Build stacked bar chart

matte_palette = [
    "#FF8A3D",  # orange-peach     (≈ 24°)
    "#FBC02D",  # golden yellow   (≈ 43°)
    "#3BB273",  # emerald green   (≈ 148°)
    "#34C6D9",  # cyan-teal       (≈ 187°)
    "#4B71FF",  # royal blue      (≈ 227°)
    "#9B6EF3",  # violet-indigo   (≈ 260°)
    "#FF5E7E",  # pink-magenta    (≈ 348°)
]
fig = px.bar(
    df_top50_detail,
    y="supplier_name",
    x="total_value",
    color="agency",
    text="value_m",
    orientation="h",
    title="Top 50 suppliers – stacked by agency",
    labels={"supplier_name": "Supplier", "total_value": "Total Value"},
    template="apple",
    color_discrete_sequence=matte_palette,   # <-- new line
)

# ------------------------------------------------------------------
# 2) Trace-level tweaks
fig.update_traces(
    texttemplate="%{text}",              # value_m already has units/format
    textposition="outside",
    marker_line_width=0,
)

# ------------------------------------------------------------------
# 3) Layout polish
fig.update_yaxes(
    title=None,
    categoryorder="total ascending"
)

fig.update_layout(
    title=dict(
        text="Top 50 Suppliers – Stacked by Agency",
        x=0.02, xanchor="left"
    ),
    xaxis_title="Total Value",
    bargap=0.18,
    legend_title_text="Agency",
    legend_traceorder="normal",
    legend_yanchor="top",
    legend_y=0.98,
    legend_xanchor="left",
    legend_x=0.78,
    margin=dict(l=160, r=40, t=60, b=40),
    height=1000,
    width=1600
)

fig.show()

In [16]:
import plotly.express as px

# Sort the summary dataframe by total_value in descending order and select the top 50 suppliers
top_50_suppliers = summary.sort_values(by='total_value', ascending=False).head(50)

# Create an interactive bar chart
fig = px.bar(
    top_50_suppliers,
    y='supplier_name',
    x='total_value',
    text='total_value',
    title='Top 50 Suppliers by Total Value',
    labels={'supplier_name': 'Supplier Name', 'total_value': 'Total Value'},
)

# Update layout for better readability
fig.update_layout(
    xaxis_tickangle=-45,
    yaxis_title='Supplier Name',
    xaxis_title='Total Value',
    height=1000,
    width=1600,
)

fig.update_yaxes(categoryorder='total ascending')

# Show the chart
fig.show()